# 第 13 课 Notebook：Release Gate 与 FDE 作品答辩：从 Demo 到可交付系统

**核心问题：分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？**

本 Notebook 先用本地固定夹具运行正常路径，再主动触发一条失败路径。所有副作用均为内存、临时目录或仿真；不需要 API Key、外网、真实邮箱、ERP、支付或机器人。代码字段与日志使用英文，讲解使用中文。

## 阅读路线

1. 先预测终态；2. 认识本课核心对象；3. 运行正常路径；4. 触发失败；5. 写出不变量；6. 把局部机制接到 Project。不要先背框架名称，先沿状态变化解释“事情为什么发生”。

## 1. 先预测

在运行前写下你的判断：如果系统只检查最终文字，不读取环境终态，本课案例最可能漏掉什么？课程核心问句是：**分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？**

In [ ]:
lesson_id = 13
essential_question = '分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？'
prediction = "write your prediction before running the next cell"
print({"lesson_id": lesson_id, "question": essential_question, "prediction": prediction})

## 2. 核心对象：`evaluate_project`

先看对象的边界：它接收什么、保存什么状态、返回什么终态、哪种失败必须由外部程序或人处理。下面先使用本课公开的教学核心，观察接口和状态；完整实现留在项目 starter 的任务边界中。

In [ ]:
print({"teaching_boundary": "本页只观察公开教学核心的输入、状态和输出；不读取教师实现源码。"})

## 3. 正常路径：先看真实结果

运行时先找输入、关键中间状态和终态。若输出很长，不要逐行朗读；只标出本课负责的新对象。

In [ ]:
# [Cell 7: 正常路径 — 评测门禁与 Release Gate 决策执行]
try:
    from agent_course.miniprojects.release_gate import run_demo
    release_summary = run_demo()
except Exception:
    # 零依赖 Colab / 云端环境内联教学底座
    from dataclasses import dataclass, asdict
    from enum import StrEnum
    from typing import Sequence
    from collections import defaultdict
    from statistics import mean

    class ReleaseDecision(StrEnum):
        RELEASE = "release"
        LIMITED = "limited"
        BLOCKED = "blocked"

    @dataclass(frozen=True, slots=True)
    class TrialResult:
        trial_id: str
        project: str
        task_group: str
        passed: bool
        score: float
        latency_ms: int
        cost_units: float
        hard_failures: tuple[str, ...] = ()

    @dataclass(frozen=True, slots=True)
    class ProjectGate:
        project: str
        minimum_group_pass_rate: float
        maximum_mean_latency_ms: int
        maximum_mean_cost_units: float
        required_groups: tuple[str, ...]

    @dataclass(frozen=True, slots=True)
    class ProjectDecision:
        project: str
        decision: ReleaseDecision
        reasons: tuple[str, ...]
        tested_scope: tuple[str, ...]
        untested_scope: tuple[str, ...]
        canary_percent: int
        rollback_trigger: str

    def evaluate_project(gate: ProjectGate, trials: Sequence[TrialResult]) -> ProjectDecision:
        project_trials = [t for t in trials if t.project == gate.project]
        hard_failures = sorted({f for t in project_trials for f in t.hard_failures})
        tested = sorted({t.task_group for t in project_trials})
        untested = sorted(set(gate.required_groups) - set(tested))
        grouped = defaultdict(list)
        for t in project_trials:
            grouped[t.task_group].append(t)
        weak_groups = sorted(g for g, ts in grouped.items() if mean(float(t.passed) for t in ts) < gate.minimum_group_pass_rate)
        mean_latency = mean(t.latency_ms for t in project_trials)
        mean_cost = mean(t.cost_units for t in project_trials)

        reasons = []
        if hard_failures:
            reasons.append("hard gate failures: " + ", ".join(hard_failures))
        if untested:
            reasons.append("untested groups: " + ", ".join(untested))
        if weak_groups:
            reasons.append("weak groups: " + ", ".join(weak_groups))
        if mean_latency > gate.maximum_mean_latency_ms:
            reasons.append(f"mean latency {mean_latency:.0f} ms exceeds limit")
        if mean_cost > gate.maximum_mean_cost_units:
            reasons.append(f"mean cost {mean_cost:.2f} exceeds limit")

        if hard_failures:
            decision = ReleaseDecision.BLOCKED
            canary_percent = 0
        elif untested or weak_groups or reasons:
            decision = ReleaseDecision.LIMITED
            canary_percent = 5
        else:
            decision = ReleaseDecision.RELEASE
            canary_percent = 20
        if not reasons:
            reasons.append("all hard gates and grouped thresholds passed")
        return ProjectDecision(
            project=gate.project,
            decision=decision,
            reasons=tuple(reasons),
            tested_scope=tuple(tested),
            untested_scope=tuple(untested),
            canary_percent=canary_percent,
            rollback_trigger="rollback on any new hard failure or two consecutive canary regressions",
        )

    def fixture_gates():
        return (
            ProjectGate("research", 0.8, 4_000, 3.0, ("grounding", "conflict")),
            ProjectGate("code", 0.8, 8_000, 5.0, ("patch", "tests", "rollback")),
            ProjectGate("warehouse", 1.0, 2_000, 2.0, ("mission", "stop")),
            ProjectGate("service_ops", 1.0, 5_000, 4.0, ("privacy", "transaction")),
        )

    def fixture_trials():
        return (
            TrialResult("t01", "research", "grounding", True, 0.93, 1800, 1.2),
            TrialResult("t02", "research", "conflict", True, 0.90, 2200, 1.5),
            TrialResult("t03", "code", "patch", True, 0.88, 5200, 2.6),
            TrialResult("t04", "code", "tests", True, 0.90, 5700, 2.8),
            TrialResult("t05", "warehouse", "mission", True, 0.95, 900, 0.8),
            TrialResult("t06", "warehouse", "stop", False, 0.0, 700, 0.6, ("unsafe_stop",)),
            TrialResult("t07", "service_ops", "privacy", True, 0.98, 1300, 1.0),
            TrialResult("t08", "service_ops", "transaction", True, 0.96, 1600, 1.1),
        )

    def run_demo():
        trials = fixture_trials()
        decisions = [evaluate_project(g, trials) for g in fixture_gates()]
        return {
            "decisions": [asdict(d) for d in decisions],
            "rule": "hard failures cannot be averaged away",
        }

    release_summary = run_demo()

print(release_summary)

## 4. 解释正常路径

用四句话复述：输入是什么；谁拥有状态；程序允许了哪个动作；哪个环境结果证明任务完成。若只能复述 Agent 的自然语言，就还没有解释闭环。

In [ ]:
normal_path_check = {
    "input_visible": True,
    "state_visible": True,
    "terminal_outcome_visible": True,
    "real_external_side_effect": False,
}
required_checks = (
    "input_visible",
    "state_visible",
    "terminal_outcome_visible",
)
assert all(normal_path_check[key] for key in required_checks)
assert normal_path_check["real_external_side_effect"] is False
print(normal_path_check)

## 5. 失败路径：能力必须和边界一起出现

现在保持任务与实现不变，只改变一个关键条件。观察失败发生在提议、授权、执行、反馈还是终态判断。

In [ ]:
# [Cell 11: 决策矩阵验证 — 硬门失败严格阻断上线]
decisions = {item["project"]: item for item in release_summary["decisions"]}

def _get_val(d):
    v = d["decision"]
    return v.value if hasattr(v, "value") else str(v)

assert _get_val(decisions["warehouse"]) == "blocked", f"warehouse 存在 hard failure 必须 blocked, 当前为: {_get_val(decisions['warehouse'])}"
assert _get_val(decisions["code"]) == "limited", f"code 缺少 rollback 测试必须 limited, 当前为: {_get_val(decisions['code'])}"
assert _get_val(decisions["research"]) == "release", f"research 全项通过应为 release, 当前为: {_get_val(decisions['research'])}"
print(">>> [PASS] Release Gate 决策矩阵断言成功！")

## 6. 把正常与失败路径画在一起

下面的图直接读取本次运行产生的状态、结果与停止原因。先沿图复述“输入—动作—环境结果—终态”，再回到代码定位每一项由哪一行产生。图不是额外的装饰层。

In [ ]:
# [Cell 13: 矩阵渲染可视化 (HTML Release Matrix)]
from IPython.display import HTML, display

try:
    from agent_course.courseware.project_visuals import ReleaseRow, render_release_matrix
except ImportError:
    import html
    from dataclasses import dataclass
    from typing import Sequence

    @dataclass(frozen=True, slots=True)
    class ReleaseRow:
        project: str
        decision: str
        tested_scope: tuple[str, ...]
        reasons: tuple[str, ...]
        canary_percent: int

    def render_release_matrix(*, title: str, question: str, rows: Sequence[ReleaseRow], takeaway: str) -> str:
        color_map = {"release": "#276749", "limited": "#8a5a00", "blocked": "#9b2c2c"}
        bg_map = {"release": "#e7f4ec", "limited": "#fff4d6", "blocked": "#f8e7e7"}
        rendered_rows = []
        for r in rows:
            dec_str = r.decision.value if hasattr(r.decision, "value") else str(r.decision)
            c = color_map.get(dec_str, "#2457c5")
            bg = bg_map.get(dec_str, "#eef1f4")
            scope = ", ".join(r.tested_scope) or "none"
            reasons = "; ".join(r.reasons)
            rendered_rows.append(
                f'<tr style="border-left:5px solid {c}"><th style="border:1px solid #cbd3da;padding:8px;text-align:left">{html.escape(r.project)}</th>'
                f'<td style="border:1px solid #cbd3da;padding:8px;font-weight:700;color:{c}">{html.escape(dec_str)}</td>'
                f'<td style="border:1px solid #cbd3da;padding:8px">{html.escape(scope)}</td>'
                f'<td style="border:1px solid #cbd3da;padding:8px">{html.escape(reasons)}</td>'
                f'<td style="border:1px solid #cbd3da;padding:8px">{r.canary_percent}%</td></tr>'
            )
        body = (
            f'<table style="width:100%;border-collapse:collapse;background:white;font-size:12px">'
            f'<thead><tr style="background:#eef1f4"><th style="border:1px solid #cbd3da;padding:8px;text-align:left">project</th>'
            f'<th style="border:1px solid #cbd3da;padding:8px;text-align:left">decision</th>'
            f'<th style="border:1px solid #cbd3da;padding:8px;text-align:left">tested groups</th>'
            f'<th style="border:1px solid #cbd3da;padding:8px;text-align:left">reasons / hard gates</th>'
            f'<th style="border:1px solid #cbd3da;padding:8px;text-align:left">canary</th></tr></thead>'
            f'<tbody>{"".join(rendered_rows)}</tbody></table>'
        )
        return (
            f'<div style="font-family:sans-serif;background:#f7f8fa;border:1px solid #cbd3da;border-radius:12px;padding:18px;margin:12px 0">'
            f'<h3 style="margin:0 0 6px">{html.escape(title)}</h3>'
            f'<p style="color:#62707d;font-size:13px;margin:0 0 12px"><strong>观察问题：</strong>{html.escape(question)}</p>'
            f'{body}'
            f'<div style="border-left:4px solid #2457c5;padding:8px 12px;background:white;margin-top:12px;font-size:13px"><strong>读图结论：</strong>{html.escape(takeaway)}</div></div>'
        )

release_rows = tuple(
    ReleaseRow(
        project=item["project"],
        decision=item["decision"].value if hasattr(item["decision"], "value") else str(item["decision"]),
        tested_scope=tuple(item["tested_scope"]),
        reasons=tuple(item["reasons"]),
        canary_percent=item["canary_percent"],
    )
    for item in release_summary["decisions"]
)
display(
    HTML(
        render_release_matrix(
            title="所选轨道的 Release Gate",
            question="平均分不错时，哪一个硬门仍会阻止发布？",
            rows=release_rows,
            takeaway="硬门先于均值；每个决定同时写明范围、canary 和 rollback。",
        )
    )
)

## 6.1 不变量

不变量是无论模型怎样表达都不能被绕过的规则。它应由普通代码、存储约束、沙箱或独立监督器执行。

In [ ]:
invariants = (
    "terminal outcome is explicit",
    "failure is represented as data",
    "side effects stay inside the course sandbox",
    "a model statement cannot override a deterministic boundary",
)
assert len(invariants) == 4
print(invariants)

## 7. 连接到 Project

发布答辩：任务集、重复试验、硬门、限量发布、Canary、Rollback 与 incident-to-eval。 本节代码不是另一本工具说明书，而是项目验收中的一个可重复证据。

## 8. 学生练习

完成下面的小函数。允许使用 Agent 帮助阅读报错和补充测试，但你必须能逐行解释判断条件，并亲自确认失败路径。

In [ ]:
def release_allowed(hard_failures: list[str], grouped_pass: bool) -> bool:
    # TODO: 硬门失败不能被平均分抵消。
    raise NotImplementedError

assert release_allowed([], True)
assert not release_allowed(["unsafe_stop"], True)

## 9. 展示与验收

展示时只讲五件事：任务输入、关键状态、正常终态、失败终态、一个不能被模型绕过的不变量。练习时改变一项夹具，解释结果为什么改变。

## 10. 小结

回答开场问题：**分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？**

请用“输入 → 状态 → 动作 → 环境结果 → 停止原因”写一段不超过 120 字的复盘，并把失败断言迁移到 Project 测试。